Author: Alana Pooler
<br>
Purpose: Complete final project

# Final Project

In this project, we will be using MLlib to build an elastic net model for predicting power consumption in tetouan city. The elastic net model will be fit using preprocessing steps applied through an MLlib pipeline, including one-hot encoding, binarization, and principle component analysis. We will also use cross validation to select optimal parameter values for the model. 

We will then use structured streaming to fit our model and generate  predictions on new data from incoming csv files. 

First, we need to import the necessary libraries and create a spark session.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer,
    Binarizer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    PCA
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# initialize spark session
spark = SparkSession.builder.getOrCreate()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57524)
Traceback (most recent call last):
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  

We need to read in the 'power_ml_data' file using pandas and convert it to a spark data frame.

We will use the Power_Zone_3 variable as our response variable and all of the other variables as predictors.

In [24]:
# read in as pandas df
pdf = pd.read_csv("data/power_ml_data.csv")

# convert to spark df and view first few rows
df = spark.createDataFrame(pdf)
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Let's look at the column names and the data types of each column.

In [25]:
df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### Part 1: Linear Regression Model

Before we can fit our model, we need to define some transformations, which we will put into a pipeline using MLlib. 

First, we need to cast the Hour column as `DoubleType` since it is currently stored as `LongType`.

We will also rename the response variable, Power_zone_3, to 'label' within the same SQLTransformer() call.

In [26]:
sql_transformer = SQLTransformer(
    statement="""
    SELECT
        *,
        CAST(Hour AS DOUBLE) AS Hour_double,
        Power_Zone_3 as label
    FROM __THIS__
    """
)

Now we need to binarize the Hour column based on the column being less than 6.5 or not, which will essentially give us an indicator of night and day.

In [27]:
hour_bin = Binarizer(
    threshold = 6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

We also want to one-hot encode the Month column. First we can use StringIndexer() to map the column values to numeric indices from 0 to 11, and then we can one-hot encode the mapped values.

In [28]:
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_ohe"]
)

Next we want to run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns.

We will first use VectorAssembler() to put these variables into one column that we can use with the PCA() estimator.

In [36]:
# place variables to run PCA fit on in one column
pca_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="pca_features"
)

# run PCA
pca = PCA(
    k=2,
    inputCol="pca_features",
    outputCol="pca_output"
)

Now we can use VectorAssembler() to put all of our predictors into one 'features' column.

In [37]:
assembler = VectorAssembler(
    inputCols=[
        "pca_output",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe"
    ],
    outputCol="features"
)

Now that we have defined all of the transformations, we can build our elastic net model using a pipeline to apply all of the transformations.

In [38]:
# define linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# build pipeline
pipeline = Pipeline(stages=[
    sql_transformer,
    hour_bin,
    month_indexer,
    month_encoder,
    pca_assembler,
    pca,
    assembler,
    lr
])

Now we need to define the parameter grid and grid values. We will test all combinations of the values 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1 for each of our two model parameters. The parameters are:
* `regParam`: controls the amount of regularization applied to the model. 
* `elasticNetParam`: controls the mix between L1 (Lasso) and L2 (Ridge) regularization.
    * A value of 0 only uses L2 regularization, a value of 1 only uses L1 regularization, and a value between 0 and 1 uses a mix of both.

In [39]:
grid_values = [0, 0.05, 0.1, 0.25, 0.5,
               0.75, 0.9, 0.95, 0.98,
               0.99, 1]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, grid_values)
    .addGrid(lr.elasticNetParam, grid_values)
    .build()
)

Next we will fit the model using 5-fold cross validation with RMSE as the criterion. Using cross validation will help us identify what values of each parameter results in the best model.

In [ ]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    parallelism=4,
    numFolds=5
)

cv_model = cv.fit(df)

Now we can look at the optimal parameter values for this model, as well as the CV error, which is the average RMSE over the 5 folds.

The optimal regularization parameter is 0.1 and the optimal elastic net parameter is 0.05. Our CV error

In [35]:
# retrieve best model
best_model = cv_model.bestModel.stages[-1]

# print parameters and CV error
print("Best regParam:", best_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_model._java_obj.getElasticNetParam())
print("CV Error:", min(cv_model.avgMetrics))

Best regParam: 0.1
Best elasticNetParam: 0.25
CV Error: 2175.179403662935


Next we will find the training set RMSE by using the fitted model as a transformer and evaluating on the entire training set.

This RMSE is about the same as the cross validation error.

In [18]:
preds_cv = cv_model.transform(df)
RegressionEvaluator().evaluate(preds_cv)

2147.0973815931757

The values we are predicting are large (in the tens of thousands), so the RMSE is quite large as well. Let's look at the predictions, actual values, and the residuals (observed - predicted) to get a better idea of how close the model is getting to the actual values.

The predicted values aren't perfect, but they aren't too far off either. Most of the residuals are between 1000 and 2000 higher than the actual values.

In [19]:
results = preds_cv.withColumn(
    "residual",
    col("label") - col("prediction")
)

results.select(
    "label",
    "prediction",
    "residual"
).show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20877.937081726264|-636.9732217262645|
|20131.08434|  18660.3633292201|   1470.7210107799|
|19668.43373| 18204.92955666827|1463.5041733317303|
|18899.27711|17590.856602658896| 1308.420507341103|
|18442.40964|16997.562545926743|1444.8470940732586|
|18130.12048|16517.979368595385|1612.1411114046168|
|17945.06024|16093.563942464634|1851.4962975353646|
|17459.27711|15723.041398159372|1736.2357118406271|
|17025.54217|15271.419222111514| 1754.122947888487|
|16794.21687|14938.751195097318|1855.4656749026817|
+-----------+------------------+------------------+
only showing top 10 rows


### Part 2: Streaming

Now we are going to read in a stream in the form of .csv files. First we need to setup the schema for the stream and the readStream.

In [20]:
schema = df.schema

stream_df = (
    spark.readStream
    .schema(schema)
    .option("header", True)
    .csv("stream_output/")
)

Now, we will use our stream and the model transformer we defined to get predictions from the incoming data. We will create a residual column with these predictions like we did in part 1.

In [21]:
pred_stream = cv_model.transform(stream_df)

pred_stream = pred_stream.withColumn(
    "residual",
    col("label") - col("prediction")
)

pred_stream = pred_stream.select(
    "label",
    "prediction",
    "residual"
)

Next we will use another transformation on the stream to rename the response variable to 'label'. Then we will join this stream with the stream we made above.

In [22]:
label_stream = stream_df.withColumnRenamed(
    "Power_Zone_3",
    "label"
)

joined_stream = pred_stream.join(
    label_stream,
    on="label",
    how ="inner"
)

Lastly, we will write the stream to the console using the `append` output mode and start the query.

After running this cell, we will use the console to run our data_production.py file. This will take a random sample of 5 rows from 'power_streaming_data.csv' and write it to a csv file in the '/stream_output' folder. Then the stream will read in that csv file containing the random sample, fit the elastic net model, and return a data frame containing the label, prediction, residual, and the predictor columns. This will be repeated 15 times on different random samples of data from 'power_streaming_data.csv'.

In [23]:
query = (
    joined_stream.writeStream
    .outputMode("append")
    .format("console")
    .start()
)

26/04/30 11:24:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6645f052-07fa-4482-abb4-08a0e57b89cb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 11:24:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12214.46809|11504.055483258104|  710.4126067418965|      21.22|   57.04|     4.925|                299.6|        220.6| 31484.63895| 25674.27386|   10|  13|
|14705.40216|19090.212187273344|-4384.8100272733445|      12.29|   65.95|     0.097|                0.864|        0.868| 40748.28897| 34203.12979|   12|  17|
|15391.37896| 10514.40195479096|  4876.977005209041|      25.18|   41.08|     4.929|                730.0|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11671.73252| 12603.27376353593| -931.5412435359303|      24.22|   67.87|     4.923|                608.4|         81.3| 34232.29759| 22989.21162|   10|  13|
|19626.01824|20954.738968905378|-1328.7207289053767|      20.25|   69.13|     0.074|                0.084|          0.1| 44964.55142| 25700.41494|   10|  20|
|15149.03226|  14080.1972490741| 1068.8350109259009|      13.18|    86.8|     4.911|                0.033|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|29627.07692|30150.027745375144| -522.9508253751446|      23.98|   46.66|     4.919|                3.097|        2.679| 48953.64238| 26775.46778|    6|  20|
|7635.054022| 7991.678160170747|-356.62413817074685|      11.27|   51.37|     0.077|                14.62|        11.69| 24942.96578| 20594.04725|   12|   8|
|16411.56923| 18616.13255294053|-2204.5633229405285|      23.14|    72.5|     0.071|                666.6|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 31288.3682|30858.242372891887|  430.1258271081133|      33.46|   31.95|     4.912|                811.0|        65.61|  42291.0299| 27603.79747|    7|  14|
|9564.984802| 8557.723660524352| 1007.2611414756484|      22.49|   58.69|     0.084|                 0.94|         0.73| 26033.43545| 17066.39004|   10|   7|
|19229.09091|16155.051422471332| 3074.0394875286675|      16.33|    85.8|     0.066|                 84.3|       

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18032.56281| 16702.04481758578| 1330.5179924142212|      11.78|   51.92|     0.085|                441.9|        477.1|  33717.9661| 19119.75684|    2|  14|
|15429.81818| 15466.59222961106|-36.774049611060036|       14.4|    86.4|      0.07|                0.084|        0.137| 23790.22605| 13113.23829|    4|   3|
|9692.196879|12692.505222270594|-3000.3083432705953|      14.04|   69.95|     0.086|                36.35|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9231.212485|11015.064061462612|-1783.8515764626118|       17.0|   57.84|     0.081|                355.7|        242.3| 31318.63118| 25866.83032|   12|  15|
|    18816.0|17682.366222929744| 1133.6337770702557|      24.19|   48.81|      0.08|                449.8|        193.6| 32271.25828| 18939.29314|    6|  15|
|17950.44534| 19359.00823570672|-1408.5628957067202|      18.62|    80.6|      0.07|                 88.4|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|23328.90282|25435.707215451974|-2106.8043954519744|      28.45|   34.58|     4.905|                810.0|        41.43| 40192.14206| 26477.29673|    8|  12|
|13694.45783|12370.202753345236|  1324.255076654763|      14.91|    72.8|     0.073|                0.055|        0.204|  20664.3038|  12423.1003|    1|   3|
|11971.53769|12056.572303056228| -85.03461305622841|       10.7|    92.2|     0.078|                0.051|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15131.81273|15307.965285677348|-176.15255567734857|      15.51|    73.8|     0.084|                0.022|        0.167| 35522.43346| 28429.57963|   12|  20|
|12443.71808|12482.824126390775| -39.10604639077428|      22.58|    74.8|     4.917|                563.1|         73.9| 30819.82301| 24780.87318|    9|  10|
|24329.37238|26654.545235028818| -2325.172855028817|       24.0|   55.86|     4.915|                0.088|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24246.80162|24141.904526405477| 104.89709359452172|      22.95|   60.04|     0.079|                0.077|        0.096| 41446.81967| 25668.11146|    5|  22|
|15504.45141|24039.152586092667| -8534.701176092667|      24.87|    91.4|     4.918|                146.0|        114.1| 36490.65483|  22759.4509|    8|  10|
|9220.668693| 9065.358234105714| 155.31045889428606|      17.91|   67.54|     0.086|                26.67|      